In [1]:
%cd ..
from claimbuster.adv_transformer.core.utils.flags import FLAGS


/home/adamj/factcheck-podcasts/src


In [2]:
import os
# display current working directory
os.getcwd()

'/home/adamj/factcheck-podcasts/src'

In [3]:
FLAGS.cs_model_dir = "/home/adamj/factcheck-podcasts/src/claimbuster/output/bba/"

In [4]:
from claimbuster.adv_transformer.core.api.api_wrapper import ClaimSpotterAPI
claimspotter = ClaimSpotterAPI()

2023-05-06 02:39:41.923403: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
[nltk_data] Downloading package punkt to /home/adamj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/adamj/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package tagsets to /home/adamj/nltk_data...
[nltk_data]   Package tagsets is already up-to-date!
[nltk_data] Downloading package stopwords to /home/adamj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Dependencies Loaded.


2023-05-06 02:39:44.008114: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcuda.so.1
2023-05-06 02:39:44.018480: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:923] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2023-05-06 02:39:44.018511: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1733] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA GeForce RTX 4090 computeCapability: 8.9
coreClock: 2.52GHz coreCount: 128 deviceMemorySize: 23.99GiB deviceMemoryBandwidth: 938.86GiB/s
2023-05-06 02:39:44.018526: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcudart.so.11.0
2023-05-06 02:39:44.030685: I tensorflow/stream_executor/platform/default/dso_loader.cc:53] Successfully opened dynamic library libcublas.so.11
2023-05-06 02:39:44.030785: I tensorflow/stream_execut

In [5]:
sentence_list = [
    'Donald Trump is the 45th President of the United States',
    'I really like cheese',
    'McDonalds earns $10 billion dollars each minute'
]

In [6]:
claimspotter.batch_sentence_query(sentence_list)

2023-05-06 02:39:48.555095: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:176] None of the MLIR Optimization Passes are enabled (registered 2)
2023-05-06 02:39:48.560495: I tensorflow/core/platform/profile_utils/cpu_utils.cc:114] CPU Frequency: 3187195000 Hz


array([[0.75899322, 0.24100678],
       [0.91082659, 0.08917341],
       [0.03764426, 0.96235574]])

In [7]:
import requests
podcasts = requests.get("http://127.0.0.1:8008/api/podcasts/")
podcasts = podcasts.json()

In [8]:
# get the uuid field of each segmentation object in each segmentation_set for each transcription in transcription_set and each audioitem in audioitem_set and each podcast in podcasts
segmentation_uuids = []
for podcast in podcasts:
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == 'spaCy':
                    segmentation_uuids.append(segmentation['uuid'])
len(segmentation_uuids)


300

In [9]:
for seg_uuid in segmentation_uuids:
    segments = requests.get(f"http://127.0.0.1:8008/api/segmentations/{seg_uuid}/")
    segments = segments.json()
    sentence_list = [utt["text_coref"] if utt.get("text_coref") else utt["text"] for utt in segments["utterance_set"]]
    scores = claimspotter.batch_sentence_query(sentence_list)

    for i, segment in enumerate(segments["utterance_set"]):
        if segment.get("text_coref"):
            requests.post(f"http://127.0.0.1:8008/api/classifications/{segment['uuid']}/", json={
                "utterance": segment["uuid"],
                "qualifier": "Checkworthiness",
                "category": "Checkworthy",
                "label": str(scores[i][1]),
                "agent": {"PROLIFIC_PID": "ClaimBuster-BBA-(COREF)"}
            })

Token indices sequence length is longer than the specified maximum sequence length for this model (1830 > 512). Running this sequence through the model will result in indexing errors


## Get Checkworthiness from Factiverse API

In [10]:
def get_fv_checkworthiness(text, language="en"):
    query = {'lang': language, 'logging':False, 'text':text}
    response = requests.post('https://api.factiverse.no/v1/claim_detection', json=query)
    response = response.json()
    scores = [resp["score"] for resp in response["detectedClaims"]]
    score = 0 if len(scores) == 0 else sum(scores)/len(scores)
    return score


In [11]:
for podcast in podcasts:
    language = podcast['language'][0:2] if podcast['language'] != "nb" else "no"
    print(podcast['title'], language)
    if language == "no":
        continue
    for audioitem in podcast['audioitem_set']:
        for transcription in audioitem['transcription_set']:
            for segmentation in transcription['segmentation_set']:
                if segmentation['name'] == 'spaCy':
                    seg_uuid = segmentation['uuid']
                    segmentation = requests.get(f"http://127.0.0.1:8008/api/segmentations/{seg_uuid}/")
                    segmentation = segmentation.json()

                    for segment in segmentation["utterance_set"]:
                        already_classified = False
                        if segment["classification_set"]:
                            for classification in segment["classification_set"]:
                                if classification["qualifier"] == "Checkworthiness" and classification["agent"] == "Factiverse":
                                    already_classified = True
                                    break
                        if already_classified:
                            continue
                            
                        # split the segment by spaces
                        words = segment["text"].split(" ")
                        if len(words) > 2:
                            fv_score = get_fv_checkworthiness(segment["text"], language)
                        else:
                            fv_score = 0

                        requests.post(f"http://127.0.0.1:8008/api/classifications/{segment['uuid']}/", json={
                            "utterance": segment["uuid"],
                            "qualifier": "Checkworthiness",
                            "category": "Checkworthy",
                            "label": str(fv_score),
                            "agent": "Factiverse"
                        })

Verdict with Ted Cruz en
Ta Kommandoen med Geir Aker no
Misjonen med Antonsen og Golden no
Leger om livet no
Norsken, svensken og dansken no
Burde vært pensum no
Huberman Lab en


ConnectionError: HTTPSConnectionPool(host='api.factiverse.no', port=443): Max retries exceeded with url: /v1/claim_detection (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7f1689dc3fd0>: Failed to establish a new connection: [Errno -3] Temporary failure in name resolution'))